# P2 ゴールデンセットを作る

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

ソラハ電機の問い合わせ（本番 600 件・架空）から、試験に使う 120 件を層別に抜き取り、層の重み・件数と区間の幅・お手本からの漏れ・似た文の塊・データカードまでを見ます。LLM は使いません。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 層別の抜き取り ― 少ない層にも最低件数を配る
`common/sampling.py` の本物のコードです。

In [ ]:
from common.sampling import stratified_sample
show(stratified_sample, 6, 25)

## 2. 抜き取りを再現し、本番と試験の分布を比べる
同じ seed なら同じ 120 件になるか。ラベルの割合は本番とどれだけ違うか。

In [ ]:
from collections import Counter
from common import kit
g = lab("run_golden", "lab02_golden_set")
pool, golden = kit.load_jsonl("inquiries_pool.jsonl"), kit.load_jsonl("golden_inquiries.jsonl")
again = stratified_sample(pool, key=g.stratum, n=120, min_per_stratum=3, seed=7)
print("seed=7 で抜き直すと:", "同じ 120 件" if [r["id"] for r in again] == [r["id"] for r in golden] else "違う")
cp, cg = Counter(r["label"] for r in pool), Counter(r["label"] for r in golden)
rows = [[x, cp[x], kit.pct(cp[x] / len(pool)), cg[x], kit.pct(cg[x] / len(golden))] for x in g.LABELS]
print(kit.table(rows, ["ラベル", "本番", "割合", "試験", "割合"]))
dp, dg = Counter(r["difficulty"] for r in pool), Counter(r["difficulty"] for r in golden)
for d in ("easy", "rule", "multi"):
    print(f"難しさ {d:<5} 本番 {kit.pct(dp[d] / len(pool))} → 試験 {kit.pct(dg[d] / len(golden))}")

## 3. 層の重み ― 本番での割合 ÷ 試験での割合
全体の正答率を言うときは、この重みで本番の割合に戻します（P3 の「本番の割合に戻した正答率」）。

In [ ]:
w = g.strata_weights(pool, golden)
cp2, cg2 = Counter(map(g.stratum, pool)), Counter(map(g.stratum, golden))
rows = [[k.replace("|", " / "), cp2[tuple(k.split("|"))], cg2[tuple(k.split("|"))], f"{v:.2f}"]
        for k, v in sorted(w.items(), key=lambda kv: kv[1])]
print(kit.table(rows, ["層（ラベル / 難しさ）", "本番", "試験", "重み"]))
print(f"層の数 {len(w)}、重みの範囲 {min(w.values()):.2f}〜{max(w.values()):.2f}")

## 4. 件数を 4 倍にすると、区間の幅は半分になる
正答率 90% のときの Wilson の 95% 区間（`common/stats.py`）。

In [ ]:
from common import stats
rows = []
for n in (30, 120, 480, 1920):
    lo, hi = stats.wilson(round(0.9 * n), n)
    rows.append([n, kit.ci_str(lo, hi), f"±{100 * (hi - lo) / 2:.1f}"])
print(kit.table(rows, ["件数", "95% 区間", "半幅（ポイント）"]))
print("±3 ポイントで知りたいなら（p=0.9）:", stats.n_for_margin(0.03, 0.9), "件 /（p=0.5）:", stats.n_for_margin(0.03, 0.5), "件")

## 5. お手本（few-shot）からの漏れ
お手本 12 件と、用件がほぼ同じ（3 文字の Jaccard ≥ 0.8）試験の問い合わせを探します。

In [ ]:
from common.sampling import leakage
shots = kit.load_jsonl("fewshot_inquiries.jsonl")
hits = leakage(golden, shots, threshold=0.8)
print(f"お手本 {len(shots)} 件と、ほぼ同じ用件の試験の問い合わせ: {len(set(h['test'] for h in hits))} 件")
for h in hits:
    print(f"   試験 {h['test']} ≈ お手本 {h['shot']}  (J={h['jaccard']})")

## 6. 似た文の塊 ― 実効の件数は 120 より少ない
用件がほぼ同じものを 1 つの塊にまとめて数えます。

In [ ]:
show(g.near_dup_clusters, 0, 16)
print()
print("試験 120 件 →", g.near_dup_clusters(golden), "塊")

## 7. データカードを書く
ラボの本物の実行（`run_golden.py`）が書いた `out/dataset_card.md` を開きます。

In [ ]:
import io, contextlib
with contextlib.redirect_stdout(io.StringIO()):
    g.main([])
print("# lab02_golden_set/out/dataset_card.md（長い行は折り返して表示）")
for line in open("lab02_golden_set/out/dataset_card.md", encoding="utf-8").read().splitlines():
    while len(line) > 50:
        print(line[:50]); line = "    " + line[50:]
    print(line)

## 8. 例題 ― 120 件・正答率 90% の幅を手で出す
Wilson の式を一つずつ計算し、`stats.wilson` の答えと比べます。

In [ ]:
import math
n, k, z = 120, 108, 1.96               # 正答 108/120 = 90%
p = k / n
den = 1 + z * z / n                    # 1 + z²/n
center = (p + z * z / (2 * n)) / den   # 真ん中は 0.9 より少し 0.5 側へ寄る
half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
print(f"p={p:.3f}  分母={den:.4f}  真ん中={center:.4f}  半幅={half:.4f}")
print(f"手計算  [{100 * (center - half):.1f}, {100 * (center + half):.1f}]")
lo, hi = stats.wilson(k, n)
print(f"stats.wilson  [{100 * lo:.1f}, {100 * hi:.1f}]  半幅 ±{100 * (hi - lo) / 2:.1f} ポイント")
print(f"（正規近似なら ±{100 * z * math.sqrt(p * (1 - p) / n):.1f} ポイント）")